In [38]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import os

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression, Perceptron
from sklearn.metrics import mean_squared_error, accuracy_score, confusion_matrix, r2_score

## **Linear Regression**
### Data Preprocessing

In [39]:
if os.path.exists('USA_Housing.csv'):
    DATA_PATH = 'USA_Housing.csv'
else:
    DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/Housing Price Predictor/USA_Housing.csv'

housing_df = pd.read_csv(DATA_PATH)

In [ ]:
housing_df.drop(columns=['Address'], inplace=True)

In [ ]:
scale_columns = ['Avg. Area Income', 'Avg. Area House Age', 'Avg. Area Number of Rooms',
                 'Avg. Area Number of Bedrooms', 'Area Population']

In [ ]:
sta_deviations = housing_df[scale_columns].std()
means = housing_df[scale_columns].mean()

In [ ]:
scaled_df = (housing_df[scale_columns] - means)/sta_deviations
scaled_df['Price'] = housing_df['Price']

In [ ]:
scaled_df.isna().sum()

In [ ]:
print(scaled_df.duplicated().sum())

In [ ]:
scaled_df

# **Training A Model**

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(scaled_df[scale_columns],
                         scaled_df['Price'], test_size=0.2, random_state=42)

In [ ]:
for each in x_train, x_test, y_train, y_test:
  print(each.shape)

In [ ]:
linear_model = LinearRegression()
train_model = linear_model.fit(x_train, y_train)

In [ ]:
train_model.coef_

In [ ]:
train_model.intercept_

# **Evaluation And Testing**

In [ ]:
test_model = linear_model.predict(x_test)

In [ ]:
avg_squared_error = mean_squared_error(y_test, test_model)
root_mean_square = np.sqrt(avg_squared_error)

In [ ]:
score = linear_model.score(x_test, y_test)

This model explains about 91.8% of the variation in house prices. That doesn't mean 91% of individual predictions are exactly correct — it means most of the overall price pattern is captured by these 5 features, while the remaining 8% comes from factors the model doesn't account for. The actual dollar-level error per prediction is what RMSE (~$100K) tells you.

In [ ]:
scatter_plt = plt.scatter(y_test, test_model, c=test_model, cmap='viridis')
plt.title('Actual vs Predicted Price')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.colorbar()

# **Locally Weighted Regression**
Since there is no specific model for locally weighted regression, to implement it we need to do some part manually.

**How will we manually implement Locally Weighted Regression (LWR)?**

We will pick 3-4 data points i.e., specific houses and will assign them weights using w⁽ⁱ⁾ = exp(−(x⁽ⁱ⁾−x)²/2τ²). Then we will fit a separate weighted regression for each data point using 2-3 different τ values. Then, in the end, we will compare if a smaller τ overfit to nearby points or a larger τ looks like a plain linear regression.

In [ ]:
random.seed(42)
indices = random.sample(range(0,len(x_train)), 4)
data_points = x_train.iloc[indices]

In [ ]:
for i in range(4):
  distance = x_train - data_points.iloc[i]
  square_distance = np.square(distance)
  final_distance = np.sum(square_distance, axis=1)
  weight = np.exp(-final_distance / (2 * np.square(1)))
  lwr = LinearRegression()
  lwr_train = lwr.fit(x_train, y_train, sample_weight=weight)
  pred = lwr_train.predict(data_points.iloc[[i]])
  print(f'Predicted: {pred[0]}      Actual: {y_train.iloc[indices[i]]}')

Predictions are way better than plain linear regression model as in linear
regression model was off by 100K USD, while here it is off by 30K to 40K USD which is
a real improvement. But four data points are not enough to be so confident
about the model.

Now we will test the model on the entire test set so that we can compare its
RMSE with plain linear regression to get the better sense of who performs
better: plain linear regression or locally weighted regression.

Predictions is a one dimensional (1D) list that holds all predictions of test set.

In [ ]:
predictions = []
for i in range(len(x_test)):
  distance = x_train - x_test.iloc[i]
  square_distance = np.square(distance)
  final_distance = np.sum(square_distance, axis=1)
  weight = np.exp(-final_distance / (2 * np.square(1)))
  lwr = LinearRegression()
  lwr_train = lwr.fit(x_train, y_train, sample_weight=weight)
  pred = lwr_train.predict(x_test.iloc[[i]])
  predictions.append(pred[0])

### **LWR Evaluation**
Since there is no standard model in sklearn for Locally Weighted Regression, we cannot use .score() method to calculate its accuracy. Instead we will use r2_score metrics to determine accuracy

In [ ]:
lwr_mean_error = mean_squared_error(y_test, predictions)
lwr_root_error = np.sqrt(lwr_mean_error)
lwr_score = r2_score(y_test, predictions)

In [ ]:
print(lwr_mean_error, lwr_root_error, lwr_score)

### **LWR vs Plain Linear Regression (Comparison)**

For this dataset, there is no major performance gap between the two models. LWR is better-suited to non-linear relationships, where the underlying pattern between features and price curves rather than following a straight line. Since the relationship in this dataset is largely linear, LWR just adds an extra layer of complexity — extra computation and no reusable model — while giving essentially the same accuracy as plain linear regression.

In [ ]:
print('Comparison between Plain Linear Regression vs Locally Weighted Regression')
print()

comparison = {
    'LWR': [lwr_root_error, lwr_score],
    'Plain': [root_mean_square, score]
}
plain_vs_lwr = pd.DataFrame(comparison, index=['RMSE', 'Score'])

print(plain_vs_lwr)
print()
print('As can be seen in the table, locally weighted regression and plain\nlinear regression perform almost identically. There is no major gap\nin their prediction results.')

# **Logistic Regression**
### Classifying House As Expensive (1) or Not Expensive (0)

In [ ]:
median_price = scaled_df['Price'].median()
scaled_df['> Median'] = (scaled_df['Price'] > median_price).astype(int)
new_xtrain, new_xtest, new_ytrain, new_ytest = train_test_split(scaled_df[scale_columns],
scaled_df['> Median'], test_size=0.2, random_state=42)

In [ ]:
classify_model = LogisticRegression()
classify_train_model = classify_model.fit(new_xtrain, new_ytrain)

# **Evaluation And Testing**

In [ ]:
class_preds = classify_train_model.predict(new_xtest)
class_score = accuracy_score(new_ytest, class_preds)
class_matrix = confusion_matrix(new_ytest, class_preds)
print('Confusion Matrix:')
print(class_matrix)

# **Perceptron**
It is a historical classifier that came before Logistic Regression. It classifies things into one of two classes — bluntly, with no in-between. It uses a direct rule of updating the weights only when the current prediction went wrong, to fix it. Unlike Logistic Regression, it does not use any probability or "chance" — just a hard yes/no decision.

In [ ]:
percep_model = Perceptron()
percep_train_model = percep_model.fit(new_xtrain, new_ytrain)

# **Evaluation And Testing**

In [ ]:
percep_preds = percep_train_model.predict(new_xtest)
percep_score = accuracy_score(new_ytest, percep_preds)

In [ ]:
print(f'Logistic Regression Score: {class_score}')
print(f'Perceptron Score:          {percep_score}')

Logistic Regression (0.905) slightly outperforms Perceptron (0.841), though both work on the same fundamental structure of θᵀx. Logistic Regression uses a smooth probability estimate to decide the class, while Perceptron makes a blunt, direct cutoff — no probability involved. This likely explains the gap: probability-based boundaries tend to generalize a bit better than a hard cutoff, especially near the decision boundary where cases are ambiguous.

# **GLM Tie-Together**

Throughout this project, linear regression and logistic regression looked structurally similar even though they solved different problems — one predicting an exact price, the other a yes/no classification. This isn't a coincidence: both are special cases of a broader framework called the Generalized Linear Model (GLM). Every prediction problem starts with an assumption about how y is distributed given x. Linear regression assumes y follows a Gaussian (normal) distribution — reasonable for a continuous value like Price, which can vary smoothly with some noise around the model's prediction. Logistic regression assumes y follows a Bernoulli distribution — reasonable when the outcome is binary, like "expensive or not." Both Gaussian and Bernoulli belong to a mathematical family called the exponential family, and GLM shows that any distribution in this family produces a hypothesis function and gradient update following the exact same recipe. That's why the training update rule looked nearly identical in both models throughout this project — not two unrelated algorithms, but one framework (GLM) applied with two different assumptions about the kind of output being predicted.

# **Project Write-Up: House Price Predictor — End-to-End**

**Goal:** Build an end-to-end machine learning project using a single housing dataset for both regression and classification.

## **Part 1 — Linear Regression**

Predicted exact house Price using 5 scaled features (Income, House Age, Rooms, Bedrooms, Population).

- RMSE ≈ $100,444
- R² = 0.918 — the model explains about 92% of the variation in house prices

## **Part 2 — Locally Weighted Regression**

Tested whether fitting a separate, neighborhood-specific model per prediction would outperform one global line.

- RMSE ≈ $101,066
- R² = 0.917 — essentially identical to Part 1

**Conclusion:** This dataset's underlying relationship is largely linear, so local weighting added computational cost (slower, no reusable model) without improving accuracy. Plain linear regression is the better practical choice here.

## **Part 3 — Logistic Regression**

Reframed the problem as classification — is a house "expensive" (above median price) or not.

- Accuracy: 90.5%
- Confusion matrix showed balanced, non-systematic errors (44 false positives, 51 false negatives out of 1000 test houses)

## **Part 4 — Perceptron**

Same classification task, using a simpler, probability-free classifier.

- Accuracy: 84.1% — lower than Logistic Regression's 90.5%

Consistent with the idea that a smooth, probability-based decision boundary generalizes better than Perceptron's hard cutoff, especially for borderline cases.

## **Part 5 — GLM**

Tied the two regression/classification approaches together conceptually — both Linear and Logistic Regression are special cases of the Generalized Linear Model framework, differing only in their assumed distribution of y (Gaussian for continuous Price, Bernoulli for binary classification). This is why their training update rules shared the same underlying structure throughout the project.

## **Overall Takeaway**

A single dataset supported the full arc of classical supervised learning techniques — regression, weighted regression, classification, and a simpler linear classifier — all traceable back to one shared mathematical foundation (dot products, gradients, and the GLM framework).

The most valuable practical lesson: added model complexity (LWR) isn't automatically better — it has to earn its place with real accuracy gains, and here it didn't.